# AAI-540-GROUP3 : AWS CloudTrail Suspicious Activity Classifier

# 1. Install libraries

In [16]:
!python --version

Python 3.12.14


In [3]:
# Install
# %pip install sagemaker 
# %pip install --upgrade boto3 botocore awscli

# 2. Setup Datalake
## 2.1 Create s3 bucket

In [17]:
import boto3
from sagemaker.core.helper.session_helper import Session
session = boto3.session.Session()
region = session.region_name
sagemaker_session = Session()
bucket = sagemaker_session.default_bucket()
s3 = boto3.Session().client(service_name="s3", region_name=region)


In [18]:
print("The bucket: {}".format(bucket))

The bucket: sagemaker-us-east-1-551414857569


In [19]:
s3_bucket_path = "s3://{}/aai-540-group3-project".format(bucket)
print("S3 bucket path is : {}".format(s3_bucket_path))

S3 bucket path is : s3://sagemaker-us-east-1-551414857569/aai-540-group3-project


## 2.2 Download dataset

In [6]:
# !curl -O https://summitroute.com/downloads/flaws_cloudtrail_logs.tar

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  240M  100  240M    0     0  39.8M      0  0:00:06  0:00:06 --:--:-- 44.5M


In [20]:
### Extract files
!mkdir -p dataset/
!tar -xvf flaws_cloudtrail_logs.tar -C dataset

flaws_cloudtrail_logs/
flaws_cloudtrail_logs/flaws_cloudtrail00.json.gz
flaws_cloudtrail_logs/flaws_cloudtrail01.json.gz
flaws_cloudtrail_logs/flaws_cloudtrail02.json.gz
flaws_cloudtrail_logs/flaws_cloudtrail03.json.gz
flaws_cloudtrail_logs/flaws_cloudtrail04.json.gz
flaws_cloudtrail_logs/flaws_cloudtrail05.json.gz
flaws_cloudtrail_logs/flaws_cloudtrail06.json.gz
flaws_cloudtrail_logs/flaws_cloudtrail07.json.gz
flaws_cloudtrail_logs/flaws_cloudtrail08.json.gz
flaws_cloudtrail_logs/flaws_cloudtrail09.json.gz
flaws_cloudtrail_logs/flaws_cloudtrail10.json.gz
flaws_cloudtrail_logs/flaws_cloudtrail11.json.gz
flaws_cloudtrail_logs/flaws_cloudtrail12.json.gz
flaws_cloudtrail_logs/flaws_cloudtrail13.json.gz
flaws_cloudtrail_logs/flaws_cloudtrail14.json.gz
flaws_cloudtrail_logs/flaws_cloudtrail15.json.gz
flaws_cloudtrail_logs/flaws_cloudtrail16.json.gz
flaws_cloudtrail_logs/flaws_cloudtrail17.json.gz
flaws_cloudtrail_logs/flaws_cloudtrail18.json.gz
flaws_cloudtrail_logs/flaws_cloudtrail19.json.

In [21]:
!gunzip dataset/flaws_cloudtrail_logs/*.json.gz
!ls -ltr dataset/flaws_cloudtrail_logs/*

-rw-r--r--. 1 sagemaker-user users  94041123 Oct  9  2020 dataset/flaws_cloudtrail_logs/flaws_cloudtrail01.json
-rw-r--r--. 1 sagemaker-user users 102617805 Oct  9  2020 dataset/flaws_cloudtrail_logs/flaws_cloudtrail00.json
-rw-r--r--. 1 sagemaker-user users  95084208 Oct  9  2020 dataset/flaws_cloudtrail_logs/flaws_cloudtrail02.json
-rw-r--r--. 1 sagemaker-user users  98074328 Oct  9  2020 dataset/flaws_cloudtrail_logs/flaws_cloudtrail03.json
-rw-r--r--. 1 sagemaker-user users 136008020 Oct  9  2020 dataset/flaws_cloudtrail_logs/flaws_cloudtrail04.json
-rw-r--r--. 1 sagemaker-user users 136720455 Oct  9  2020 dataset/flaws_cloudtrail_logs/flaws_cloudtrail05.json
-rw-r--r--. 1 sagemaker-user users 133047147 Oct  9  2020 dataset/flaws_cloudtrail_logs/flaws_cloudtrail06.json
-rw-r--r--. 1 sagemaker-user users 134071678 Oct  9  2020 dataset/flaws_cloudtrail_logs/flaws_cloudtrail07.json
-rw-r--r--. 1 sagemaker-user users 139298702 Oct  9  2020 dataset/flaws_cloudtrail_logs/flaws_cloudtrail

## 2.3 Copy the dataset to s3 bucket 

In [14]:
import boto3
import os

# Set up the S3 client
s3_client = boto3.client('s3')

# Define the local directory containing your files
local_directory = 'dataset/flaws_cloudtrail_logs'

# Set your S3 bucket and destination path
s3_bucket = 'sagemaker-us-east-1-551414857569'
s3_prefix = 'aai-540-group3-project'

# Iterate over all files in the local directory
for root, dirs, files in os.walk(local_directory):
    for file_name in files:
        # Construct the full file path
        file_path = os.path.join(root, file_name)

        # Construct the relative S3 key (path in S3)
        s3_key = os.path.join(s3_prefix, file_name)
        
        # Upload the file
        s3_client.upload_file(file_path, s3_bucket, s3_key)
        print(f"Uploaded {file_name} to s3://{s3_bucket}/{s3_key}")

Uploaded flaws_cloudtrail00.json to s3://sagemaker-us-east-1-551414857569/aai-540-group3-project/flaws_cloudtrail00.json
Uploaded flaws_cloudtrail01.json to s3://sagemaker-us-east-1-551414857569/aai-540-group3-project/flaws_cloudtrail01.json
Uploaded flaws_cloudtrail02.json to s3://sagemaker-us-east-1-551414857569/aai-540-group3-project/flaws_cloudtrail02.json
Uploaded flaws_cloudtrail03.json to s3://sagemaker-us-east-1-551414857569/aai-540-group3-project/flaws_cloudtrail03.json
Uploaded flaws_cloudtrail04.json to s3://sagemaker-us-east-1-551414857569/aai-540-group3-project/flaws_cloudtrail04.json
Uploaded flaws_cloudtrail05.json to s3://sagemaker-us-east-1-551414857569/aai-540-group3-project/flaws_cloudtrail05.json
Uploaded flaws_cloudtrail06.json to s3://sagemaker-us-east-1-551414857569/aai-540-group3-project/flaws_cloudtrail06.json
Uploaded flaws_cloudtrail07.json to s3://sagemaker-us-east-1-551414857569/aai-540-group3-project/flaws_cloudtrail07.json
Uploaded flaws_cloudtrail08.json

In [15]:
!aws s3 ls $s3_bucket_path/

2026-09-24 15:58:37  102617805 flaws_cloudtrail00-checkpoint.json
2026-09-24 15:58:20  102617805 flaws_cloudtrail00.json
2026-09-24 15:58:20   94041123 flaws_cloudtrail01.json
2026-09-24 15:58:21   95084208 flaws_cloudtrail02.json
2026-09-24 15:58:22   98074328 flaws_cloudtrail03.json
2026-09-24 15:58:22  136008020 flaws_cloudtrail04.json
2026-09-24 15:58:23  136720455 flaws_cloudtrail05.json
2026-09-24 15:58:24  133047147 flaws_cloudtrail06.json
2026-09-24 15:58:25  134071678 flaws_cloudtrail07.json
2026-09-24 15:58:26  139298702 flaws_cloudtrail08.json
2026-09-24 15:58:27  135375522 flaws_cloudtrail09.json
2026-09-24 15:58:28  130795014 flaws_cloudtrail10.json
2026-09-24 15:58:29  131653952 flaws_cloudtrail11.json
2026-09-24 15:58:30  135432540 flaws_cloudtrail12.json
2026-09-24 15:58:31  133126547 flaws_cloudtrail13.json
2026-09-24 15:58:32  135327927 flaws_cloudtrail14.json
2026-09-24 15:58:33  133478616 flaws_cloudtrail15.json
2026-09-24 15:58:34  134283237 flaws_cloudtrail16.json